# Study Assistant: backend server
Runs the AI model for your website. Runtime > Change runtime type > **T4 GPU**, then run cells 1 to 4 in order.
**Leave cell 4 running** while people use the site. Stop it and the site goes offline.

In [ ]:
# CELL 1: install
!pip -q install flask flask-cors transformers accelerate bitsandbytes requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
# CELL 2: load the model (first run downloads it)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-3B-Instruct"   # fast. For better quality use "Qwen/Qwen2.5-7B-Instruct" (slower)
tok = AutoTokenizer.from_pretrained(MODEL)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
SYS = "You are an exam-preparation assistant. Use ONLY the study text provided. Never add facts that are not in the text."

def ask(prompt, max_new_tokens=1500, system=None):
    msgs = [{"role": "system", "content": system or SYS}, {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.3, top_p=0.9)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
print("Model ready")

In [ ]:
# CELL 3: start the API server
import threading
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__); CORS(app); lock = threading.Lock()

@app.get("/health")
def health(): return {"ok": True, "model": MODEL}

@app.post("/generate")
def generate():
    d = request.get_json(force=True)
    with lock:                                   # one request at a time on the GPU
        try: text = ask(d["prompt"], min(int(d.get("max_tokens", 1500)), 2500), d.get("system"))
        except Exception as e: return jsonify(error=str(e)), 500
    return jsonify(text=text)

threading.Thread(target=lambda: app.run(port=5000, threaded=True), daemon=True).start()
print("Server started on port 5000")

In [ ]:
# CELL 4: make it public, update the website, and keep running
#@title Optional: let Colab update config.json in your GitHub repo automatically
GITHUB_REPO = ""  #@param {type:"string"}
import subprocess, re, time, base64, json, requests

p = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:5000", "--no-autoupdate"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in p.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: url = m.group(0); break
threading.Thread(target=lambda: [None for _ in p.stdout], daemon=True).start()   # keep the pipe drained
time.sleep(5)
print("PUBLIC SERVER URL:", url)
print("Health check:", requests.get(url + "/health").text)

if GITHUB_REPO:
    from getpass import getpass
    token = getpass("GitHub token (fine-grained, Contents: read and write on that repo): ")
    api = f"https://api.github.com/repos/{GITHUB_REPO}/contents/config.json"
    h = {"Authorization": f"Bearer {token}"}
    r = requests.get(api, headers=h); sha = r.json().get("sha") if r.ok else None
    body = {"message": "Update server URL", "content": base64.b64encode(json.dumps({"server": url}).encode()).decode()}
    if sha: body["sha"] = sha
    print("config.json updated on GitHub" if requests.put(api, headers=h, json=body).ok else "GitHub update failed. Paste the URL into config.json by hand.")
else:
    print('Put this in config.json on GitHub:  {"server": "' + url + '"}')

print("\nServer is live. Keep this cell running.")
while True: time.sleep(30)